# Aula 9 — Do Texto aos Números

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

Este notebook não usa biblioteca nenhuma além do que já vem no Python.
Todo o tokenizador cabe em umas quarenta linhas.

## Parte A: Demonstração

### O corpus: onze obras de Machado de Assis

São 3,7 milhões de caracteres em domínio público, baixados do Project
Gutenberg. A grafia é a de 1899, e isso vai importar.

In [ ]:
import collections
import json
import re
import urllib.request

# Endereço dos dados desta aula no GitHub.
URL_CORPUS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/machado.txt"
# Alternativa para testar offline:
# URL_CORPUS = "../../data/machado.txt"

if URL_CORPUS.startswith("http"):
    corpus = urllib.request.urlopen(URL_CORPUS).read().decode("utf-8")
else:
    corpus = open(URL_CORPUS, encoding="utf-8").read()

print(f"{len(corpus):,} caracteres")
print(f"{len(corpus.split()):,} palavras")
print()
print(corpus[1200:1700])

### O que um LLM faz: prever o próximo pedaço

Não existe uma resposta certa. Existe uma distribuição de probabilidade
sobre todas as continuações possíveis.

In [ ]:
seguintes = collections.Counter(re.findall(r"\bminha mãe (\w+)", corpus, re.I))
total = sum(seguintes.values())

print(f'"minha mãe ___" aparece {total} vezes, com {len(seguintes)} continuações:')
for palavra, quantas in seguintes.most_common(8):
    print(f"  {palavra:<8} {quantas / total:>5.0%}")

### As três unidades possíveis

Por palavra o vocabulário explode. Por letra o texto fica longo demais.

In [ ]:
# O mesmo padrão que o tokenizador usa para separar o texto em pedaços.
PADRAO = re.compile(r"\s*[^\s\W\d_]+|\s*\d|\s*[^\s\w]|\s+")

pedacos = PADRAO.findall(corpus)

print(f"por letra:   {len(corpus):>9,} peças, vocabulário de {len(set(corpus)):>6,}")
print(f"por palavra: {len(pedacos):>9,} peças, vocabulário de {len(set(pedacos)):>6,}")

### O BPE, em uma frase

Encontre o par de peças vizinhas mais comum, junte as duas numa peça
nova, e repita. Comece pela versão mínima, com `banana`.

In [ ]:
def mostrar_bpe(texto, quantos_passos):
    simbolos = list(texto)
    print(f"começo: {' '.join(simbolos)}")
    for passo in range(quantos_passos):
        pares = collections.Counter(zip(simbolos, simbolos[1:]))
        par = max(pares, key=pares.get)
        novo = "".join(par)
        saida, i = [], 0
        while i < len(simbolos):
            if i < len(simbolos) - 1 and (simbolos[i], simbolos[i + 1]) == par:
                saida.append(novo)
                i += 2
            else:
                saida.append(simbolos[i])
                i += 1
        simbolos = saida
        print(f"passo {passo + 1}: junta {novo!r} ({pares[par]}x) -> {' '.join(simbolos)}")

mostrar_bpe("banana banana banana", 3)

### O BPE de verdade: sobre bytes

Todo texto do mundo é uma sequência de bytes, e existem 256 valores
possíveis. Começando por eles, nenhum caractere fica de fora.

Repare que `ã` ocupa dois bytes: no começo do treino, ele é duas peças.

In [ ]:
print("'ã' em bytes:", list("ã".encode("utf-8")))
print("'a' em bytes:", list("a".encode("utf-8")))
print("'🙂' em bytes:", list("🙂".encode("utf-8")))

### O treino do tokenizador

Duas funções: uma conta os pares, a outra aplica uma fusão. O laço de
treino chama as duas, uma vez por fusão.

Para ir rápido, o algoritmo agrupa as palavras iguais e conta cada uma
pelo número de vezes que ela aparece, em vez de percorrer o texto todo.

In [ ]:
def contar_pares(palavras):
    pares = collections.Counter()
    for simbolos, quantas in palavras:
        for par in zip(simbolos, simbolos[1:]):
            pares[par] += quantas
    return pares


def aplicar_fusao(simbolos, a, b, novo):
    saida, i = [], 0
    while i < len(simbolos):
        if i < len(simbolos) - 1 and simbolos[i] == a and simbolos[i + 1] == b:
            saida.append(novo)
            i += 2
        else:
            saida.append(simbolos[i])
            i += 1
    return saida


def treinar_bpe(texto, tamanho_vocabulario):
    frequencia = collections.Counter(PADRAO.findall(texto))
    palavras = [[list(p.encode("utf-8")), n] for p, n in frequencia.items()]

    fusoes = []
    for novo in range(256, tamanho_vocabulario):
        pares = contar_pares(palavras)
        if not pares:
            break
        a, b = max(pares, key=pares.get)
        fusoes.append((a, b))
        for item in palavras:
            if a in item[0]:
                item[0] = aplicar_fusao(item[0], a, b, novo)
    return fusoes


# Treina num pedaço do corpus, para caber no tempo da aula.
fusoes = treinar_bpe(corpus[:300_000], 512)
print(f"{len(fusoes)} fusões")

In [ ]:
# A tabela: que sequência de bytes cada número representa.
tabela = {i: bytes([i]) for i in range(256)}
for indice, (a, b) in enumerate(fusoes):
    tabela[256 + indice] = tabela[a] + tabela[b]

for numero in list(range(256, 268)) + list(range(500, 512)):
    pedaco = tabela[numero].decode("utf-8", errors="replace")
    print(f"  {numero}: {pedaco!r}")

### Codificar e decodificar

Codificar aplica as fusões **na mesma ordem** em que o treino as
aprendeu. Decodificar só cola os bytes de volta.

In [ ]:
ordem = {par: 256 + i for i, par in enumerate(fusoes)}


def codificar(texto):
    saida = []
    for pedaco in PADRAO.findall(texto):
        simbolos = list(pedaco.encode("utf-8"))
        while len(simbolos) >= 2:
            candidatos = [p for p in zip(simbolos, simbolos[1:]) if p in ordem]
            if not candidatos:
                break
            a, b = min(candidatos, key=lambda p: ordem[p])
            simbolos = aplicar_fusao(simbolos, a, b, ordem[(a, b)])
        saida.extend(simbolos)
    return saida


def decodificar(numeros):
    bruto = b"".join(tabela[n] for n in numeros)
    return bruto.decode("utf-8", errors="replace")


frase = "O menino era pai do homem."
numeros = codificar(frase)
print(numeros)
print([decodificar([n]) for n in numeros])
print(repr(decodificar(numeros)))

### O lote de treino

O alvo é a entrada andada uma casa. Uma janela de 9 tokens dá 8 exemplos
de treino, não um só.

In [ ]:
janela = codificar("O menino era pai do homem")[:9]

print(f"{'entrada':<38} {'alvo'}")
for corte in range(1, len(janela)):
    entrada = decodificar(janela[:corte])
    alvo = decodificar([janela[corte]])
    print(f"{entrada!r:<38} {alvo!r}")

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conhecendo o corpus

Rode a célula e observe: quantos caracteres diferentes existem, e quais
são os mais comuns?

In [ ]:
contagem = collections.Counter(corpus)
print(f"{len(contagem)} caracteres diferentes")
for caractere, quantas in contagem.most_common(8):
    print(f"  {caractere!r:<6} {quantas:>8,}")

In [ ]:
if len(corpus) > 3_000_000:
    print(f"✅ {len(corpus):,} caracteres carregados.")
else:
    print("❌ Confira se a célula que baixa o corpus rodou até o fim.")

### Exercício 2: contando pares

Crie `meus_pares`, um `Counter` com a contagem de pares de letras
vizinhas na frase abaixo. Use `zip(frase, frase[1:])`.

In [ ]:
frase_curta = "o rato roeu a roupa do rei de roma"

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(meus_pares.most_common(5))

In [ ]:
if meus_pares.most_common(1)[0][0] == (" ", "r"):
    print("✅ O par mais comum é espaço seguido de 'r', 5 vezes.")
    print("   Em texto de verdade, o espaço entra em quase toda fusão do começo.")
else:
    print("❌ Confira: o par mais comum deveria ser (' ', 'r').")

### Exercício 3: aplicando uma fusão

Use `aplicar_fusao` para juntar o par `(114, 111)`, que é `r` e `o`, na
lista de bytes de `frase_curta`. Guarde em `depois_da_fusao`.

O número do símbolo novo é 256, o primeiro que sobra depois dos bytes.

In [ ]:
antes_da_fusao = list(frase_curta.encode("utf-8"))
print(f"antes: {len(antes_da_fusao)} símbolos")

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"depois: {len(depois_da_fusao)} símbolos")
print(depois_da_fusao)

In [ ]:
if len(depois_da_fusao) == len(antes_da_fusao) - 3:
    print("✅ Os três 'ro' da frase viraram um símbolo cada: 3 peças a menos.")
else:
    print(f"❌ Esperava {len(antes_da_fusao) - 3} símbolos e vieram {len(depois_da_fusao)}.")

### Exercício 4: treinando um BPE pequeno

Treine um tokenizador com vocabulário de **320** (ou seja, 64 fusões)
nos primeiros 200.000 caracteres do corpus. Guarde em `minhas_fusoes`.

Leva uns poucos segundos.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
minha_tabela = {i: bytes([i]) for i in range(256)}
for indice, (a, b) in enumerate(minhas_fusoes):
    minha_tabela[256 + indice] = minha_tabela[a] + minha_tabela[b]

for numero in range(256, 256 + len(minhas_fusoes)):
    print(f"  {numero}: {minha_tabela[numero].decode('utf-8', 'replace')!r}")

In [ ]:
if len(minhas_fusoes) == 64:
    print("✅ 64 fusões: de 256 a 319.")
    print("   Repare que quase todas são pares de duas letras muito comuns.")
else:
    print(f"❌ Esperava 64 fusões e vieram {len(minhas_fusoes)}.")

### Exercício 5: codificando uma frase sua

Rode e observe. Troque a frase por uma sua, inclusive com acento e
emoji, e veja que nada quebra.

O emoji vira quatro tokens, um por byte. Cada byte sozinho não forma um
caractere, então a lista mostra losangos. Junte os quatro de volta e o
emoji reaparece inteiro: é o que a última linha faz.

In [ ]:
minha_frase = "Não sei se a senhora é boa cousa. 🙂"
meus_numeros = codificar(minha_frase)

print(f"{len(minha_frase)} caracteres viraram {len(meus_numeros)} tokens")
print(meus_numeros)
print([decodificar([n]) for n in meus_numeros])
print(repr(decodificar(meus_numeros)))

In [ ]:
if decodificar(codificar(minha_frase)) == minha_frase:
    print("✅ Codificar e decodificar devolveu exatamente a frase original.")
else:
    print("❌ A ida e volta mudou o texto. Confira as duas funções.")

### Exercício 6: medindo a compressão

Calcule quantos caracteres, em média, cabem em um token. Use a fatia
`corpus[:100_000]` e guarde o resultado em `caracteres_por_token`.

$$\text{caracteres por token} = \frac{\text{quantidade de caracteres}}{\text{quantidade de tokens}}$$

In [ ]:
fatia = corpus[:100_000]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"{caracteres_por_token:.2f} caracteres por token")

In [ ]:
if 1.5 < caracteres_por_token < 3.5:
    print(f"✅ {caracteres_por_token:.2f} caracteres por token.")
    print("   Com 512 tokens dá menos que com 1.024: mais fusões comprimem mais.")
else:
    print("❌ Confira se você dividiu os caracteres pelos tokens, nessa ordem.")

### Exercício 7: desafio, o par entrada/alvo

Monte as listas `entrada` e `alvo` a partir de `janela_longa`. O alvo é a
entrada andada **uma** casa para a esquerda, e as duas têm o mesmo
tamanho.

Dica: se a janela tem 17 tokens, a entrada usa os 16 primeiros e o alvo
usa os 16 últimos.

In [ ]:
janela_longa = codificar(corpus[5029:5140])[:17]
print(f"{len(janela_longa)} tokens na janela")

In [ ]:
# SEU CODIGO AQUI

In [ ]:
for posicao in range(4):
    print(f"vendo {decodificar(entrada[:posicao + 1])!r:<30} "
          f"tem que prever {decodificar([alvo[posicao]])!r}")

In [ ]:
if len(entrada) == len(alvo) == 16 and entrada[1:] == alvo[:-1]:
    print("✅ 16 exemplos de treino saíram de uma janela de 17 tokens.")
else:
    print("❌ Confira: entrada é janela_longa[:-1] e alvo é janela_longa[1:].")

Agora, em texto: o corpus desta aula é de 1899. Escreva duas ou três
frases sobre o que o modelo vai aprender de errado por causa disso, e
como você resolveria. Edite esta célula (duplo clique nela) e escreva sua
resposta no lugar deste parágrafo.